# Draga LLM
## What Must an Inference State Remember?
### Sharp Memory Bounds for Softmax Attention and Exact Recurrent Checkpoints

All code for this study will be released at [DatarConsulting/llm-inference-state](https://github.com/DatarConsulting/llm-inference-state).

**Publication notebook · CPU · English.** In Colab choose **Runtime → Run all**. The last cell downloads **Draga_LLM_arXiv.zip** for the arXiv source-upload form. It also provides the public preprint, mathematical supplement and anonymous review PDF. No model download, GPU, access token or Drive mount is needed for the default run.

The paper asks what an inference engine must transmit at a restart when the next query is unknown. This notebook reconstructs its mathematical figures, its archived BF16 decoding measurements and its offline FP32 census. The default input includes observations for all 2,592 heads and native words for a stratified sample of 162 heads.

| Evidence | What this notebook recalculates |
|---|---|
| Spherical attention | Rational suffix witness, finite dimension bounds, harmonic shell contributions, finite designs and rotations |
| KV cache | Throughput from 24 timing pairs, KV reservation, raw fallbacks and greedy token agreement |
| Recurrent checkpoint | Word round trips, optimal references, a two-format byte stream, complete-census totals and plots |

## 1. Choose replay scope

The default is self-contained. Full word-level replay requires **Bitwise_State_Full_Data.zip** and around 1 GB of temporary disk space. To enable it, select FULL_WORD_REPLAY and provide an existing file path or choose the archive when prompted. Every run creates a new working directory.

In [ ]:
#@title Replay and delivery settings
FULL_WORD_REPLAY = False #@param {type:"boolean"}
FULL_DATA_PATH = "" #@param {type:"string"}
DOWNLOAD_ARXIV_ZIP = True #@param {type:"boolean"}

import base64, hashlib, importlib.util, io, os, shutil, subprocess, sys, tempfile
from pathlib import Path
from zipfile import ZipFile
from IPython.display import Markdown, FileLink, Image, display
in_colab = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
base = Path("/content") if in_colab else Path.cwd()
work = Path(tempfile.mkdtemp(prefix="inference_state_",dir=base))
source, evidence, release = work/"source", work/"evidence", work/"publication"
source.mkdir()
evidence.mkdir()
release.mkdir()
os.environ["MPLBACKEND"]="Agg"
os.environ["PYTHONDONTWRITEBYTECODE"]="1"
print("Run directory:", work)

## 2. Prepare the CPU runtime

NumPy, SciPy and Matplotlib handle the calculations. Zstandard 0.25.0 reproduces the archived compression totals. A C++17 compiler builds the row codec; pdfLaTeX typesets each edition. Colab downloads missing system packages on a new runtime. A local installation can run offline.

In [ ]:
from importlib.metadata import version
def run(*args, **kwargs):
    subprocess.run([str(a) for a in args],check=True,**kwargs)

packages=[n for n in ("numpy","scipy","matplotlib") if importlib.util.find_spec(n) is None]
if importlib.util.find_spec("zstandard") is None or version("zstandard")!="0.25.0":
    packages.append("zstandard==0.25.0")
if packages:
    run(sys.executable,"-m","pip","install","-q",*packages)
def tex_ready():
    wanted=["article.cls","lineno.sty","environ.sty","mathtools.sty","microtype.sty","t1lmtt.fd"]
    return shutil.which("pdflatex") and shutil.which("kpsewhich") and all(
        subprocess.run(["kpsewhich",n],stdout=subprocess.DEVNULL).returncode==0 for n in wanted)
if not tex_ready() or not shutil.which("g++") or not shutil.which("pdftoppm"):
    if not in_colab:
        raise RuntimeError("Install TeX Live with latex-extra, recommended fonts, lmodern, g++ and poppler-utils.")
    run("apt-get","-qq","update")
    run("apt-get","-qq","install","-y","texlive-latex-extra","texlive-fonts-recommended","lmodern","g++","poppler-utils")
print("Python",sys.version.split()[0],"/",", ".join(f"{n} {version(n)}" for n in ("numpy","scipy","matplotlib","zstandard")))

## 3. Restore the publication inputs

The folded cells hold the manuscript, code, recorded observations and a compressed raw-word sample. They include **no pre-rendered publication figures**. Checksums identify the released inputs. After extraction, edit body.tex for the article or checkpoint.tex for the bit theorem, if desired. Later cells rebuild from these files.

In [ ]:
#@title English manuscript, code and observations (1/1)
core_blob = bytearray()
core_blob.extend(base64.b64decode(
'...'
))

In [ ]:
#@title Native-word sample (1/4)
word_blob = bytearray()
word_blob.extend(base64.b64decode(
'...'
))

In [ ]:
#@title Native-word sample (2/4)
word_blob.extend(base64.b64decode(
'...'
))

In [ ]:
#@title Native-word sample (3/4)
word_blob.extend(base64.b64decode(
'...'
))

In [ ]:
#@title Native-word sample (4/4)
word_blob.extend(base64.b64decode(
'...'
))

In [ ]:
assert hashlib.sha256(core_blob).hexdigest()=="1c4499db753738b836af33f3ecfaf443bd1860aac2aad0f8d7eba026663b1db2"
assert hashlib.sha256(word_blob).hexdigest()=="19233a2ccb6c40392e221e0d37da0cce486b45ca8fe39544cf340cb3c79fa48c"
with ZipFile(io.BytesIO(core_blob)) as bundle:
    bundle.extractall(source)
(source/"anc/words/data/heads.u32.gz").write_bytes(word_blob)
del core_blob, word_blob
(source/"figures").mkdir()
anc=source/"anc"
for line in (anc/"inputs.sha256").read_text().splitlines():
    digest, filename=line.split("  ",1)
    assert hashlib.sha256((anc/filename).read_bytes()).hexdigest()==digest, filename
print("Release inputs and archived Lean files verified.")
print("Article:",source/"body.tex")
print("Bit proof:",source/"checkpoint.tex")

## 4. Reconstruct the mathematical figures

The first panel starts with two zero-valued prefixes. One common append gives a gap of \(4/45\). The second panel plots the Fourier eigenvalue contributions to the entropy sum on the circle. Its ordinate illustrates the proof; practical cache sizes are measured separately. The second mathematical figure evaluates the finite streaming-dimension inequalities.

The run also checks a two-sign identity, finite design examples, a rotating moment recurrence, a shared-head error box, exact interval optimization and native edge cases. The general asymptotics have written proofs in the supplement.

In [ ]:
run(sys.executable,anc/"reproduce.py","theory",evidence)
display(Image(filename=str(evidence/"entropy/checkpoint.png"),width=850))

## 5. Replay the GPU record

The original BF16 run used one RTX PRO 6000 Blackwell, two books, three prefix lengths and four alternating timing pairs per condition. The analyzer verifies the input hash, divides pooled tokens by pooled time, and reconstructs the reserved KV bytes. The 384 positions belong to six distinct continuations. This cell reanalyzes the recorded session.

In [ ]:
import csv, json
run(sys.executable,anc/"reproduce.py","gpu-record",evidence)
with (evidence/"kv/pooled.csv").open() as stream:
    for row in csv.DictReader(stream):
        print(f"{int(row['n']):>7,} tokens | dense {float(row['fi']):.2f} | compact {float(row['compact']):.2f} tok/s | {float(row['gain']):+.2f}%")
display(Image(filename=str(evidence/"kv/budget.png"),width=780))

## 6. Re-encode and decode recurrent states

Each head contains 16,384 native FP32 words. The replay checks every selected head against its input. It then writes a new stream in which a one-byte selector precedes the shorter complete frame, reads it back and verifies each digest. The default raw sample covers every trace, layer and position. Full published totals always use the 2,592-head census. To repeat every original word comparison, choose the full replay in section 1.

In [ ]:
args=[sys.executable,anc/"reproduce.py","words",evidence]
if FULL_WORD_REPLAY:
    if FULL_DATA_PATH.strip():
        full_path=Path(FULL_DATA_PATH).expanduser().resolve()
    elif in_colab:
        from google.colab import files
        uploaded=files.upload()
        if len(uploaded)!=1:
            raise ValueError("Select exactly one full recurrent-data ZIP.")
        name=next(iter(uploaded))
        full_path=work/Path(name).name
        full_path.write_bytes(uploaded.pop(name))
    else:
        raise ValueError("Set FULL_DATA_PATH to the full recurrent-data ZIP.")
    args.extend(["--full-data",full_path])
run(*args)
subset="full" if FULL_WORD_REPLAY else "sample"
print("Raw replay in this run:",(evidence/subset/"mixed/mixed_result.json").read_text())
print("Full census totals:",(evidence/"census.json").read_text())

## 7. Inspect all eight vector figures

The previews below are rendered from the newly generated PDFs. The manuscript uses the vector files. Each plotting program is under anc, and its inputs and output tables can be inspected in the run directory.

In [ ]:
figure_names=["checkpoint","budget","dimension","reference_cost","layers","mode_competition","paired","fallback"]
for name in figure_names:
    pdf=source/"figures"/(name+".pdf")
    preview=evidence/(name+"_preview")
    run("pdftoppm","-scale-to","900","-f","1","-l","1","-singlefile","-png",pdf,preview)
    display(Markdown("**"+name.replace("_"," ").capitalize()+"**"),Image(filename=str(preview)+".png",width=720))

## 8. Compile the source ZIP

The builder stores main.tex, its section files, the conference style, figures, tables and ancillary evidence. It opens the actual archive in a clean directory, checks the CRCs and compiles all three editions from that extraction. The public source ZIP names the author. The anonymous review PDF contains the appendix and NeurIPS checklist.

In [ ]:
run(sys.executable,anc/"build.py",release)
submission=release/"Draga_LLM_arXiv.zip"
with ZipFile(submission) as archive:
    assert archive.testzip() is None
    assert set("figures/"+n+".pdf" for n in figure_names)<=set(archive.namelist())
print("Upload file:",submission.name,"|",f"{submission.stat().st_size:,} bytes")

## 9. Download

Upload **Draga_LLM_arXiv.zip** directly as the arXiv source and select **main.tex** if prompted. The default triggers one browser download in Colab. The Files pane and links give access to each PDF. For an optional new GPU session, follow anc/kv/GPU_PROTOCOL.md in a compatible environment. A new hardware run writes its results separately.

In [ ]:
for item in [submission,release/"Draga_LLM_Paper.pdf",release/"Draga_LLM_Math_Supplement.pdf",release/"Draga_LLM_Anonymous_Review.pdf"]:
    if in_colab:
        from google.colab import output
        from IPython.display import HTML
        from html import escape
        from functools import partial
        from google.colab import files
        callback="inference.download."+item.suffix[1:]+"_"+item.stem
        output.register_callback(callback,partial(files.download,str(item)))
        display(HTML(f'<button onclick="google.colab.kernel.invokeFunction(\'{callback}\', [], {{}})">{escape(item.name)}</button>'))
    else:
        display(FileLink(str(item)))
if in_colab and DOWNLOAD_ARXIV_ZIP:
    from google.colab import files
    files.download(str(submission))